# **6. Shortening protein sequences with ProtRL**

[ProtRL](https://github.com/AI4PDLab/ProtRL) repeatedly generates sequences, scores them, and updates a protein language model. Here the reward favours sequences close to 20 amino acids:

$$r(s)=-\left|20-|s|\right|$$

Iteration 1 is generated by the original model. Later iterations use the updated model, allowing us to follow the change in sequence length.

> Select **Runtime → Change runtime type → T4 GPU** before starting.


## **1. Setup**


In [ ]:
%pip uninstall -q -y torchao
%pip install -q -U transformers datasets peft accelerate trl pandas matplotlib
!git clone -q --depth 1 https://github.com/AI4PDLab/ProtRL.git /content/ProtRL


**Exercise.** Open the ProtRL repository and identify the four scripts called during each iteration.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import torch

assert torch.cuda.is_available(), "Select a GPU runtime and run again."

MODEL = "AI4PD/ProtGPT3-112M"
ITERATIONS = 6
OUTPUT = Path("/content/protrl_length")


**Exercise.** Why does six generation rounds correspond to only five observable model updates?


## **2. Inspect the reward**


In [ ]:
def length_reward(sequence, target=20):
    return -abs(target - len(sequence))

for length in [20, 40, 80, 100]:
    print(length, length_reward("A" * length))


**Exercise.** Which length receives the highest reward? Why are all other rewards negative?


## **3. Run the online loop**

Each iteration performs four operations:

1. generate 100 sequences;
2. record their lengths;
3. convert length into reward; and
4. update ProtGPT3 with ProtRL GRPO.

The first set is generated before any update and provides the baseline.


In [ ]:
!cd /content/ProtRL && bash ProtRL.sh \
  --model_dir {MODEL} \
  --output_dir {OUTPUT} \
  --max_iteration_num {ITERATIONS}


**Exercise.** Find where generation, scoring, and training begin in the printed log.


## **4. Measure the change**


In [ ]:
logs = pd.read_csv(OUTPUT / "logs.csv")
summary = logs.groupby("iteration_num")["length"].agg(["mean", "std", "min", "max"])
summary.round(1)


**Exercise.** Compare the mean and standard deviation in the first and final iterations.


In [ ]:
ax = summary["mean"].plot(marker="o", ylim=(0, None))
ax.set(xlabel="Iteration", ylabel="Mean generated length (aa)")
plt.show()

change = summary["mean"].iloc[-1] - summary["mean"].iloc[0]
print(f"Change from first to final iteration: {change:.1f} aa")


**Exercise.** Increase `ITERATIONS` to 8 in a fresh runtime. Does the mean approach 20 aa more closely?


## **5. Interpretation**

ProtRL changes the probability of the generated amino-acid tokens and of the end-of-sequence token. Rewarding shorter sequences makes earlier termination more likely.

This is deliberately a single-objective experiment. A shorter sequence is not necessarily folded, stable, or functional; practical protein design requires additional rewards or filters.
